In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import os
import sys
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import requests

In [ ]:
def download_images(image_list):
    """
    Downloads images from a list of (URL, filepath) tuples using multiple threads.
    """
    if not image_list:
        print("No images to download.")
        return

    with ThreadPoolExecutor(max_workers=8) as executor:
        results = list(tqdm(executor.map(download_single_image, image_list), total=len(image_list), desc="Downloading images"))
    
    successful_downloads = sum(results)
    print(f"\nSuccessfully downloaded {successful_downloads} of {len(results)} requested images.")

In [ ]:
# --- Configuration (Paths relative to the project root directory) ---
TRAIN_CSV_PATH = '/kaggle/input/amazon-ml-data-set/processed_data_drop_2.csv'
TEST_CSV_PATH = '/kaggle/input/amazon-ml-data-set/processed_data_joined_2_test.csv'
IMAGE_ROOT_DIR = '/kaggle/working/images'

In [ ]:
def download_single_image(args):
    """Helper function to download one image, handling potential errors."""
    url, filepath = args
    try:
        url = url.replace(' ', '%20')
        response = requests.get(url, stream=True, timeout=15)
        if response.status_code == 200:
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)
            return True
        else:
            return False
    except Exception as e:
        return False

In [ ]:
def prepare_and_download(df: pd.DataFrame, split_name: str, root_dir: str):
    """
    Prepares a list of images to download and calls the download utility.
    The images are saved using their 'sample_id' as the filename.
    """
    print(f"--- Preparing '{split_name.upper()}' dataset ---")

    # Create the destination directory (e.g., 'images/train')
    output_dir = os.path.join(root_dir, split_name)
    os.makedirs(output_dir, exist_ok=True)
    print(f"Image save directory: {output_dir}")

    download_tasks = []
    print("Scanning CSV and checking for existing images...")

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Scanning {split_name} CSV"):
        image_url = row['image_link']
        sample_id = row['sample_id']

        # Construct the final filepath, e.g., "images/train/33127.jpg"
        # Using int() ensures the ID is a clean number without decimals
        filepath = os.path.join(output_dir, f"{int(sample_id)}.jpg")

        # Only add the image to the download list if it doesn't already exist
        if not os.path.exists(filepath):
            download_tasks.append((image_url, filepath))

    if not download_tasks:
        print(f"-> All {len(df)} images for the '{split_name}' set already exist. Nothing to do.")
    else:
        print(f"-> Found {len(download_tasks)} new images to download for the '{split_name}' set.")
        download_images(download_tasks)

    print(f"--- Finished processing '{split_name.upper()}' dataset ---\n")


In [ ]:
if __name__ == '__main__':
    print("--- SCRIPT STARTED ---")
    print(f"Running from directory: {os.getcwd()}")

    try:
        train_df = pd.read_csv(TRAIN_CSV_PATH)
        test_df = pd.read_csv(TEST_CSV_PATH) # Fixed: Uncommented this line
        print("Successfully loaded train.csv and test.csv.")
    except FileNotFoundError as e:
        print(f"\n[FATAL ERROR] {e}")
        print("Please make sure you are running this script from the project's ROOT directory (e.g., AMAZON_ML_CHALLENGE).")
        sys.exit(1)

    # Run the process for both datasets
    prepare_and_download(df=train_df, split_name='train', root_dir=IMAGE_ROOT_DIR)
    prepare_and_download(df=test_df, split_name='test', root_dir=IMAGE_ROOT_DIR)

    print("--- ALL TASKS COMPLETE ---")